In [ ]:
import pandas as pd
import os
from pathlib import Path
import datetime

ROOT_DIR = Path.cwd().parent.parent.parent

lkp_folder = os.path.join(ROOT_DIR, 'external_data','lookups','EHR','primary_care')

from pomegranate.db.ukbdb import UKBDatabase
Database = UKBDatabase()

def per(numerator,denominator,digits=None):
    return str(round((numerator/denominator)*100,digits))+'%'

# Make gp_prescriptions_harmonized SQL table


In [ ]:
harmonized_cols = [
    'eid','data_provider','issue_date',
    'reference_dmd_code','clean_entry_code','entry_code_in_lkp',
    'orig_entry_code','entry_ontology'
]
harmonized_cols_i = ['entry_id'] + harmonized_cols

orig_columns_i = [
    'eid','data_provider','issue_date','read_2','bnf_code','dmd_code','drug_name','quantity','id'
]

In [ ]:
def check_out_table(table_name:str, columns:list = None, get_all=False):
    if get_all is False:
        suffix=' LIMIT 30'
    elif get_all is True:
        suffix=''
    if columns == None:
        return pd.DataFrame(Database.query(f"SELECT * FROM {table_name}{suffix};").fetchall())
    else:
        if (columns==harmonized_cols) or (columns==harmonized_cols_i):
            print('Dropping eid column')
            return pd.DataFrame(Database.query(f"SELECT * FROM {table_name}{suffix};").fetchall(), columns=columns).drop(columns=['eid'])
        else:
            return pd.DataFrame(Database.query(f"SELECT * FROM {table_name}{suffix};").fetchall(), columns=columns)

In [ ]:
def show_indexes(table_name):
    c = Database.query(f"SHOW INDEX FROM {table_name};")
    columns = [desc[0] for desc in c.description]
    return pd.DataFrame(c.fetchall(), columns=columns)

## Clean/reformat

## dm+d

1. Pull out dm+d codes
2. Join the reference codes from the lookup - make this the 'reference_dmd_code' and the original code as 'orig_entry_code' and 'clean_entry_code'. Make the 'entry_in_lkp' column a default values of 1.
3. Backfill those with dmd_code and entry_code as NULL (i.e., not in the lookup and so don't have a reference code):
    - entry_code = orig_entry_code
    - dmd_code stays NULL
    - entry_code_in_lkp = 0

In [ ]:
show_indexes('dmd_reference_map')

Reference to check against

In [ ]:
n_dmd_src = Database.query('SELECT COUNT(*) FROM gp_prescriptions WHERE dmd_code IS NOT NULL;').fetchall()[0][0]

In [ ]:
# filter for dm+d entries, merge with reference codes
print(datetime.datetime.now())
Database.query('''
DROP TABLE IF EXISTS temp_dmd_harmonized;
CREATE TABLE temp_dmd_harmonized
SELECT
    p.id,
    p.eid,
    p.data_provider,
    p.issue_date,
    m.reference_code AS 'reference_dmd_code',
    p.dmd_code as 'clean_entry_code',
    1 as 'entry_code_in_lkp',
    p.dmd_code AS 'orig_entry_code',
    'dmd' as 'entry_ontology'
FROM
    gp_prescriptions_i p
LEFT JOIN dmd_reference_map m
    ON p.dmd_code = m.dmd_code
    WHERE p.dmd_code IS NOT NULL;
''')
print(datetime.datetime.now())

TEST - check counts

In [ ]:
n_dmd_clean_1 = Database.query('SELECT COUNT(*) FROM temp_dmd_harmonized;').fetchall()[0][0]

try:
    assert n_dmd_clean_1 == n_dmd_src
except:
    print(f"N dm+d entries in gp_prescriptions: {n_dmd_src}\nN entries in dmd_harmonized: {n_dmd_clean_1}")

Update index

In [ ]:
# index
Database.query("CREATE INDEX idx_dmd_harmonized_reference_code ON temp_dmd_harmonized(reference_dmd_code);") # make where faster

In [ ]:
show_indexes('temp_dmd_harmonized')

Batches not working great, will try the original approach again and see if the time is improved with the index

In [ ]:
# set in_lkp to 0 if no reference code
print(datetime.datetime.now())
Database.query('''
UPDATE temp_dmd_harmonized 
SET entry_code_in_lkp = 0
WHERE reference_dmd_code IS NULL;
''')
print(datetime.datetime.now())

In [ ]:
# make sure the update is complete
assert Database.query("SELECT * FROM temp_dmd_harmonized WHERE reference_dmd_code IS NULL AND entry_code_in_lkp = 1;").rowcount == 0

TEST - check counts

In [ ]:
n_dmd_clean_2 = Database.query('SELECT COUNT(*) FROM temp_dmd_harmonized;').fetchall()[0][0]

try:
    assert n_dmd_clean_2 == n_dmd_src
except:
    print(f"N dm+d entries in gp_prescriptions: {n_dmd_src}\nN entries in temp_dmd_harmonized: {n_dmd_clean_2}")

## BNF entries

- Pull all BNF entries that do NOT have a dm+d code
- Update truncated codes with more specified codes in the lookup with identical descriptions
- Otherwise, map entry code to to formatted BNF code (remove . and .00)
- Set the default 'entry_code_in_lkp' to 1
- At the same time, map to dmd
- Find those with NULL entry_code and set in_lkp to 0

Reference to check against

In [ ]:
n_bnf_src = Database.query('SELECT COUNT(*) FROM gp_prescriptions WHERE bnf_code IS NOT NULL AND dmd_code IS NULL;').fetchall()[0][0]

In [ ]:
show_indexes('bnf_code_desc_expansion_map')

In [ ]:
show_indexes('bnf_dmd_map')

Pull out the BNF entries and first map them to updated codes from the lookup by desc

In [ ]:
# takes 6 minutes 
print(datetime.datetime.now())
Database.query('''
DROP TABLE IF EXISTS temp_bnf_harmonized;
CREATE TABLE temp_bnf_harmonized
SELECT 
    p.id,
    p.eid,
    p.data_provider,
    p.issue_date,
    d.reference_dmd_code AS 'reference_dmd_code',
    l.lkp_bnf AS 'clean_entry_code',
    1 AS 'entry_code_in_lkp',
    p.bnf_code AS 'orig_entry_code',
    'bnf' AS 'entry_ontology'
FROM 
    gp_prescriptions_i p
LEFT JOIN bnf_code_desc_expansion_map l
    ON l.desc = p.drug_name AND
    l.entry_bnf = p.bnf_code
LEFT JOIN bnf_dmd_map d
    ON l.lkp_bnf = d.bnf_code
WHERE
    p.bnf_code IS NOT NULL AND
    p.dmd_code IS NULL;
''')
print(datetime.datetime.now())

This can complete quickly in the notebook but the query itself actually takes a while to move the data. 

You can check the process of it if things are hanging by going into the MySQL terminal and running SHOW PROCESSLIST;

It should have a state of 'Sending data'

TEST - check counts

In [ ]:
n_bnf_clean_1 = Database.query('SELECT COUNT(*) FROM temp_bnf_harmonized;').fetchall()[0][0]

try:
    assert n_bnf_clean_1 == n_bnf_src
except:
    print(f"N BNF entries in gp_prescriptions: {n_bnf_src}\nN entries in temp_bnf_harmonized: {n_bnf_clean_1}")

In [ ]:
# add key length stipulation
Database.query("ALTER TABLE temp_bnf_harmonized MODIFY clean_entry_code VARCHAR(50);")

In [ ]:
# new indexes for the next step
# takes 11 minutes
Database.query("CREATE INDEX idx_orig_entry_code ON temp_bnf_harmonized(orig_entry_code);")
Database.query("CREATE INDEX idx_clean_entry_code ON temp_bnf_harmonized(clean_entry_code);")

For codes that didn't need to be updated by description, format and map to dm+d

In [ ]:
show_indexes('bnf_dmd_map')

In [ ]:
show_indexes('temp_bnf_harmonized')

In [ ]:
show_indexes('bnf_formatted_map')

In [ ]:
# 10 minutes
print(datetime.datetime.now())
Database.query('''
UPDATE temp_bnf_harmonized b
LEFT JOIN bnf_formatted_map f
    ON b.orig_entry_code = f.original_bnf
LEFT JOIN bnf_dmd_map d
    ON f.formatted_bnf = d.bnf_code
SET 
    b.clean_entry_code = f.formatted_bnf, 
    b.reference_dmd_code = d.reference_dmd_code
WHERE b.clean_entry_code IS NULL;
''')
print(datetime.datetime.now())

In [ ]:
Database.query("ALTER TABLE temp_bnf_harmonized MODIFY clean_entry_code VARCHAR(50);")

In [ ]:
# new index for next step
#Database.query("DROP INDEX idx_clean_entry_code ON temp_bnf_harmonized;")
#Database.query("CREATE INDEX idx_clean_entry_code ON temp_bnf_harmonized(clean_entry_code);")

Finally, indicate which are NOT in the BNF lookup

In [ ]:
show_indexes('bnf_desc_type_lkp')

In [ ]:
show_indexes('temp_bnf_harmonized')

In [ ]:
# took 5 minutes
print(datetime.datetime.now())
Database.query('''
UPDATE temp_bnf_harmonized bh
LEFT JOIN bnf_desc_type_lkp bdl
ON bh.clean_entry_code = bdl.bnf_code
SET bh.entry_code_in_lkp = 0
WHERE bdl.bnf_code IS NULL;
''')
print(datetime.datetime.now())

PATCH case where the original entry code is already in the lookup format

In [ ]:
print(datetime.datetime.now())
Database.query('''
UPDATE temp_bnf_harmonized b
JOIN bnf_dmd_map d
  ON d.bnf_code = b.orig_entry_code
SET
  b.clean_entry_code = COALESCE(b.clean_entry_code, b.orig_entry_code),
  b.reference_dmd_code = COALESCE(b.reference_dmd_code, d.reference_dmd_code)
WHERE
  b.clean_entry_code IS NULL
   OR b.reference_dmd_code IS NULL;
''')
print(datetime.datetime.now())

In [ ]:
print(datetime.datetime.now())
Database.query('''
UPDATE temp_bnf_harmonized b
JOIN bnf_desc_type_lkp l
  ON l.bnf_code = b.orig_entry_code
SET
  b.clean_entry_code  = COALESCE(b.clean_entry_code, b.orig_entry_code),
  b.entry_code_in_lkp = 1
WHERE
  b.clean_entry_code IS NULL
  OR b.entry_code_in_lkp = 0;
''')
print(datetime.datetime.now())

Test

In [ ]:
n_bnf_clean_2 = Database.query('SELECT COUNT(*) FROM temp_bnf_harmonized;').fetchall()[0][0]

try:
    assert n_bnf_clean_2 == n_bnf_src
except:
    print(f"N BNF entries in gp_prescriptions: {n_bnf_src}\nN entries in temp_bnf_harmonized: {n_bnf_clean_2}")

## Read
Have to make sure only adding read entries where the BNF is empty

By default, all codes are in the lookup so the only step here is to map to dm+d

In [ ]:
show_indexes('welsh_read_dmd_map')

In [ ]:
# make sure the read codes are read in case sensitive with BINARY - not by default
# working version 23 min
print(datetime.datetime.now())
Database.query('''
DROP TABLE IF EXISTS temp_read_harmonized;
CREATE TABLE temp_read_harmonized
SELECT 
    p.id,
    p.eid,
    p.data_provider,
    p.issue_date,
    d.dmd_code as 'reference_dmd_code',
    p.read_2 AS 'clean_entry_code',
    1 as 'entry_code_in_lkp',
    p.read_2 AS 'orig_entry_code',
    'welsh_read' AS 'drug_ontology'
FROM
    gp_prescriptions_i p
LEFT JOIN welsh_read_dmd_map d
    ON BINARY p.read_2 = BINARY d.welsh_read_code
WHERE 
    p.read_2 IS NOT NULL AND 
    p.bnf_code IS NULL AND
    p.data_provider='4';
''')
print(datetime.datetime.now())

TEST

In [ ]:
n_read_src = Database.query("SELECT COUNT(*) FROM gp_prescriptions WHERE read_2 IS NOT NULL AND bnf_code IS NULL AND data_provider='4';").fetchall()[0][0]

In [ ]:
n_read_clean = Database.query('SELECT COUNT(*) FROM temp_read_harmonized;').fetchall()[0][0]

try:
    assert n_read_clean == n_read_src
except:
    print(f"N Welsh Read entries in gp_prescriptions: {n_read_src}\nN entries in temp_read_harmonized: {n_read_clean}")

#### Update in_lkp flag

This doesn't work well in SQL due to the case sensitivity of the Read codes. Everything I've tried hasn't successfully captured the codes not in the lookup (indexes arent pulled despite forcing, which means it has to read thorugh the full table with a case sensitive query each time, and doesnt update the proper values after 240 minutes). 

This approach instead uses pandas to more quickly pull the IDs for those entries that have codes not in the lookup, then updates the proper values based on the row id

In [ ]:
from pomegranate.define import Define
read_key = Define().welsh_read_lkp.copy()
read_key.dropna(inplace=True)

In [ ]:
welsh_check = pd.DataFrame(Database.query("SELECT id, orig_entry_code FROM temp_read_harmonized;").fetchall(), columns=['entry_id', 'read_code'])
welsh_not_in_lkp = list(welsh_check[~welsh_check['read_code'].isin(read_key['read_code'])]['entry_id'])

In [ ]:
print(datetime.datetime.now())
Database.query(f'''
UPDATE temp_read_harmonized
SET entry_code_in_lkp = 0
WHERE id IN ({','.join([str(x) for x in welsh_not_in_lkp])});
''')
print(datetime.datetime.now())

In [ ]:
assert len(welsh_not_in_lkp) == Database.query("SELECT COUNT(*) FROM temp_read_harmonized WHERE entry_code_in_lkp = 0;").fetchall()[0][0]

In [ ]:
n_read_clean_1 = Database.query('SELECT COUNT(*) FROM temp_read_harmonized;').fetchall()[0][0]

try:
    assert n_read_clean_1 == n_read_src
except:
    print(f"N Welsh Read entries in gp_prescriptions: {n_read_src}\nN entries in temp_read_harmonized: {n_read_clean}")

In [ ]:
# the above query doesnt use the index anyway so it just slows it down to do it earlier
Database.query("CREATE INDEX idx_clean_read_code ON temp_read_harmonized(clean_entry_code);")

## Final check
Make sure that the number of entries in the source table == the number of entries in the new tables. and that there are no duplicates

In [ ]:
# dmd
n_dmd_clean = Database.query('SELECT COUNT(*) FROM temp_dmd_harmonized;').fetchall()[0][0]
n_dmd_src = Database.query('SELECT COUNT(*) FROM gp_prescriptions WHERE dmd_code IS NOT NULL;').fetchall()[0][0]

# bnf
n_bnf_clean = Database.query('SELECT COUNT(*) FROM temp_bnf_harmonized;').fetchall()[0][0]
n_bnf_src = Database.query('SELECT COUNT(*) FROM gp_prescriptions WHERE dmd_code IS NULL AND bnf_code IS NOT NULL;').fetchall()[0][0]

# read
n_read_clean = Database.query('SELECT COUNT(*) FROM temp_read_harmonized;').fetchall()[0][0]
n_read_src = Database.query('SELECT COUNT(*) FROM gp_prescriptions WHERE bnf_code IS NULL AND read_2 IS NOT NULL AND data_provider="4";').fetchall()[0][0]

In [ ]:
try:
    assert n_dmd_clean == n_dmd_src
    assert n_bnf_clean == n_bnf_src
    assert n_read_clean == n_read_src
except:
    print(f'''DMD
    N entries in gp_prescriptions: {n_dmd_src}
    N entries in dmd_harmonized: {n_dmd_clean}
BNF
    N entries in gp_prescriptions: {n_bnf_src}
    N entries in bnf_harmonized: {n_bnf_clean}
Read
    N entries in gp_prescriptions: {n_read_src}
    N entries in read_harmonized: {n_read_clean}''')

# Combine into single harmonized table
Leaving out lookups (descriptions, code types)

In [ ]:
SCHEMA_GP_PRESCRIPTIONS_HARMONIZED = """
DROP TABLE IF EXISTS gp_prescriptions_harmonized;
CREATE TABLE IF NOT EXISTS gp_prescriptions_harmonized (
    id INT,
    eid INT(7) UNSIGNED,
    data_provider INT(1),
    issue_date DATE,
    reference_dmd_code VARCHAR(155),
    clean_entry_code VARCHAR(155),
    entry_code_in_lkp INT,
    orig_entry_code VARCHAR(155),
    entry_ontology VARCHAR(10)
);
"""
# create table
Database.query(SCHEMA_GP_PRESCRIPTIONS_HARMONIZED)

In [ ]:
print(datetime.datetime.now())
Database.query(f'''
INSERT INTO gp_prescriptions_harmonized
(SELECT * FROM temp_dmd_harmonized)
UNION ALL
(SELECT * FROM temp_bnf_harmonized)
UNION ALL
(SELECT * FROM temp_read_harmonized)
;''')
print(datetime.datetime.now())

# Add VMP column (homogeneous codes)


In [ ]:
# prep new columns
print(datetime.datetime.now())
# add dmd type (temporary column)
Database.query('ALTER TABLE gp_prescriptions_harmonized ADD COLUMN dmd_type VARCHAR(5);')
# add VMP code
#Database.query('ALTER TABLE gp_prescriptions_harmonized DROP COLUMN reference_vmp_code;')
Database.query('ALTER TABLE gp_prescriptions_harmonized ADD COLUMN reference_vmp_code VARCHAR(155);')
print(datetime.datetime.now())

In [ ]:
# add code type
print(datetime.datetime.now())
Database.query(f'''
CREATE TEMPORARY TABLE dmd_code_type_unique AS
SELECT dmd_code, MAX(dmd_type) AS dmd_type
FROM dmd_desc_type_lkp
GROUP BY dmd_code;
''')

Database.query(f'''
CREATE INDEX ix_dctu_code ON dmd_code_type_unique(dmd_code);
''')

Database.query(f'''
UPDATE gp_prescriptions_harmonized p
JOIN dmd_code_type_unique r
  ON p.reference_dmd_code = r.dmd_code
SET p.dmd_type = r.dmd_type
WHERE p.dmd_type IS NULL;
''')
print(datetime.datetime.now())

In [ ]:
# copy code where type is vmp, otherwise map from AMP
print(datetime.datetime.now())
Database.query(f'''
UPDATE gp_prescriptions_harmonized p
LEFT JOIN dmd_amp_vmp_map m 
       ON p.reference_dmd_code = m.amp_code
SET p.reference_vmp_code = CASE 
    WHEN p.dmd_type = 'vmp' THEN p.reference_dmd_code
    WHEN p.dmd_type = 'amp' THEN m.reference_vmp_code
    ELSE NULL
END;
''')
print(datetime.datetime.now())

In [ ]:
# remove temporary column
print(datetime.datetime.now())
Database.query(f'''
ALTER TABLE gp_prescriptions_harmonized DROP COLUMN dmd_type;
''')
print(datetime.datetime.now())

## Make simplified homogeneous table

In [ ]:
Database.query("DROP TABLE IF EXISTS gp_prescriptions_homogeneous;")
Database.query(f'''
CREATE TABLE IF NOT EXISTS gp_prescriptions_homogeneous AS (
    SELECT id, eid, data_provider, issue_date, reference_vmp_code
    FROM gp_prescriptions_harmonized
);''')
Database.query("CREATE INDEX gp_presc_homo ON gp_prescriptions_homogeneous(eid, data_provider, reference_vmp_code);")

## Make simplified homogeneous table

In [ ]:
Database.query("DROP TABLE IF EXISTS gp_prescriptions_homogeneous;")
Database.query(f'''
CREATE TABLE IF NOT EXISTS gp_prescriptions_homogeneous AS (
    SELECT id, eid, data_provider, issue_date, reference_vmp_code
    FROM gp_prescriptions_harmonized
);''')
Database.query("CREATE INDEX gp_presc_homo ON gp_prescriptions_homogeneous(eid, data_provider, reference_vmp_code);")

# QC

## Tests on lengths per ontology (passed)

In [ ]:
# check lengths
n_clean = Database.query('SELECT COUNT(*) FROM gp_prescriptions_harmonized;').fetchall()[0][0]
try:
    assert n_clean == n_dmd_clean + n_bnf_clean + n_read_clean
except:
    print(f'N entries in gp_prescriptions_harmonized: {n_dmd_src}\nN entries in harmonized ontology tables: {n_dmd_clean + n_bnf_clean + n_read_clean}')

In [ ]:
# check actual row numbers
orig_entry_ids = Database.query("SELECT id FROM gp_prescriptions_i;").fetchall()
orig_entry_ids = [x[0] for x in orig_entry_ids]

harmonized_entry_ids = Database.query("SELECT id FROM gp_prescriptions_harmonized;").fetchall()
harmonized_entry_ids = [x[0] for x in harmonized_entry_ids]

In [ ]:
try: 
    assert set(orig_entry_ids) == set(harmonized_entry_ids)
except AssertionError:
    orig_entries_missing = list(set(orig_entry_ids) - set(harmonized_entry_ids))
    orig_entries_missing = [str(x) for x in orig_entries_missing]
    assert len(orig_entries_missing) == 48950

In [ ]:
entries_not_harmonized = pd.DataFrame(
    Database.query(f'''SELECT * FROM gp_prescriptions_i WHERE id IN ({", ".join(orig_entries_missing)})''').fetchall(), 
    columns=orig_columns_i)

None of the entries have codes - only descriptions

In [ ]:
entries_not_harmonized['dmd_code'].value_counts()

In [ ]:
entries_not_harmonized['bnf_code'].value_counts()

In [ ]:
entries_not_harmonized['read_2'].value_counts()

In [ ]:
entries_not_harmonized[~entries_not_harmonized['read_2'].isna()].drop(columns='eid')

Build indexes

In [ ]:
print(datetime.datetime.now())
Database.query(f'''
CREATE INDEX gp_presc_clean ON gp_prescriptions_harmonized(eid, data_provider, clean_entry_code);
CREATE INDEX gp_presc_ref_dmd ON gp_prescriptions_harmonized(eid, data_provider, reference_dmd_code);
''')
print(datetime.datetime.now())